# Construct legislation JSON (takes ~5 min)

Run this notebook when you want to refresh the v2 static database. It exports `legislation.json` from this folder; after checking it, manually replace `v2/public/legislation.json`.

## Step 1: Call the API

Fetch current Commonwealth Acts and Regulations from the Federal Register titles API. This step keeps the raw API response shape unchanged.

In [1]:
from pathlib import Path
from urllib.parse import urlencode
from urllib.error import URLError
from urllib.request import Request, urlopen
import json
import socket
import time

API_BASE_URL = 'https://api.prod.legislation.gov.au/v1/titles/search'
SEARCH_CRITERIA = "and(status(InForce),pointintime(Latest),type(Principal),collection(Act,LegislativeInstrument))"
PAGE_SIZE = 100
SELECT_FIELDS = ','.join([
    'administeringDepartments',
    'collection',
    'hasCommencedUnincorporatedAmendments',
    'id',
    'isInForce',
    'isPrincipal',
    'name',
    'number',
    'optionalSeriesNumber',
    'searchContexts',
    'seriesType',
    'subCollection',
    'year',
])
EXPAND_FIELDS = 'administeringDepartments,searchContexts($expand=fullTextVersion)'


def build_request(skip, top=PAGE_SIZE):
    params = urlencode({
        '$select': SELECT_FIELDS,
        '$expand': EXPAND_FIELDS,
        '$orderby': 'name asc',
        '$count': 'true',
        '$top': str(top),
        '$skip': str(skip),
    })
    return Request(
        f"{API_BASE_URL}(criteria='{SEARCH_CRITERIA}')?{params}",
        headers={
            'Accept': 'application/json',
            'User-Agent': 'Legification/2.0',
        },
    )


def fetch_page(request, retries=5):
    for attempt in range(1, retries + 1):
        try:
            with urlopen(request, timeout=180) as response:
                return json.loads(response.read().decode('utf-8'))
        except (TimeoutError, socket.timeout, URLError) as error:
            if attempt == retries:
                raise RuntimeError(f'API request failed after {retries} attempts: {request.full_url}') from error
            print(f'Retrying page request after {type(error).__name__} ({attempt}/{retries})')
            time.sleep(attempt * 2)


def fetch_titles():
    titles = []

    skip = 0
    total_count = None

    while True:
        payload = fetch_page(build_request(skip))

        if total_count is None:
            total_count = payload.get('@odata.count')

        values = payload.get('value') if isinstance(payload.get('value'), list) else []
        expected_values = PAGE_SIZE

        if total_count is not None:
            expected_values = min(PAGE_SIZE, total_count - skip)

        if 0 < len(values) < expected_values:
            values = []
            for small_skip in range(skip, skip + expected_values, 10):
                small_top = min(10, skip + expected_values - small_skip)
                small_payload = fetch_page(build_request(small_skip, small_top))
                small_values = small_payload.get('value') if isinstance(small_payload.get('value'), list) else []
                values.extend(small_values)

        titles.extend(values)

        if not values:
            break

        skip += PAGE_SIZE

        if total_count is not None and skip >= total_count:
            break

    return titles


raw_titles = fetch_titles()
print(f'Fetched {len(raw_titles)} titles')


Fetched 24931 titles


## Step 2: Transform the data

Keep only in-force principal Acts and Regulations, then map each title to the compact JSON shape used by the app.

In [2]:
def legislation_type(title):
    if title.get('collection') == 'Act':
        return 'Act'

    if title.get('collection') == 'LegislativeInstrument':
        return 'Legislative-instrument'

    return None


def transform_titles(raw_titles):
    records = {}

    for title in raw_titles:
        title_type = legislation_type(title)

        if not title.get('id') or not title.get('name'):
            continue

        if not title.get('isPrincipal') or not title.get('isInForce') or title_type is None:
            continue

        records[title['id']] = {
            'id': title['id'],
            'source': 'federal-register',
            'jurisdiction': 'commonwealth',
            'title': title['name'],
            'year': title.get('year'),
            'url': f"https://www.legislation.gov.au/{title['id']}",
            'type': title_type,
        }

    return sorted(records.values(), key=lambda record: record['title'])


records = transform_titles(raw_titles)
print(f'Prepared {len(records)} records')


Prepared 24931 records


## Step 3: Export the JSON

Write `legislation.json` next to this notebook and show a download link. The notebook does not update `v2/public/legislation.json` directly.

In [3]:
OUTPUT_PATH = Path('legislation.json')


def export_json(records, output_path=OUTPUT_PATH):
    output_path.write_text(
        json.dumps(records, ensure_ascii=False, separators=(',', ':')),
        encoding='utf-8',
    )

    print(f'Exported {len(records)} records to {output_path.resolve()}')

    try:
        from IPython.display import FileLink, display
    except ImportError:
        return output_path

    display(FileLink(str(output_path)))
    return output_path


export_json(records)


Exported 24931 records to C:\Users\johnn\Documents\GitHub\Legification\db_construct\legislation.json


c:\Users\johnn\Documents\GitHub\Legification\db_construct\legislation.json

WindowsPath('legislation.json')